In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
import time
from typing import List, Dict, Optional
from dataclasses import dataclass
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
import re

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@dataclass
class Article:
    """Data class for article information"""
    title: str
    content: str
    url: str
    source: str
    scores: Optional[Dict[str, float]] = None
    
    def to_dict(self):
        return {
            'title': self.title,
            'content': self.content[:500],  # Preview only
            'url': self.url,
            'source': self.source,
            **({f'score_{k}': v for k, v in self.scores.items()} if self.scores else {})
        }


class ArticleScraper:
    """
    Flexible web scraper for news and research articles
    """
    
    def __init__(self, timeout=10, max_retries=3):
        self.timeout = timeout
        self.max_retries = max_retries
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
    def scrape_url(self, url: str) -> Optional[Article]:
        """
        Scrape a single article URL
        
        Returns Article object or None if scraping fails
        """
        for attempt in range(self.max_retries):
            try:
                response = requests.get(url, headers=self.headers, timeout=self.timeout)
                response.raise_for_status()
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract title and content using multiple strategies
                title = self._extract_title(soup)
                content = self._extract_content(soup)
                
                if not title or not content:
                    logger.warning(f"Could not extract title or content from {url}")
                    return None
                
                source = urlparse(url).netloc
                
                return Article(
                    title=title,
                    content=content,
                    url=url,
                    source=source
                )
                
            except requests.RequestException as e:
                logger.error(f"Attempt {attempt + 1}/{self.max_retries} failed for {url}: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                continue
        
        return None
    
    def _extract_title(self, soup: BeautifulSoup) -> str:
        """Extract article title using multiple strategies"""
        # Strategy 1: OpenGraph meta tag
        og_title = soup.find('meta', property='og:title')
        if og_title and og_title.get('content'):
            return og_title['content'].strip()
        
        # Strategy 2: Standard title tag
        if soup.title:
            return soup.title.string.strip()
        
        # Strategy 3: h1 tag
        h1 = soup.find('h1')
        if h1:
            return h1.get_text(strip=True)
        
        return ""
    
    def _extract_content(self, soup: BeautifulSoup) -> str:
        """Extract article content using multiple strategies"""
        # Strategy 1: OpenGraph description
        og_desc = soup.find('meta', property='og:description')
        if og_desc and og_desc.get('content'):
            content = og_desc['content'].strip()
        else:
            content = ""
        
        # Strategy 2: Article tag
        article = soup.find('article')
        if article:
            paragraphs = article.find_all('p')
            content += ' ' + ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 3: Main content div (common patterns)
        if not content:
            main_content = soup.find(['div'], class_=re.compile(r'(article|content|post|entry|body)', re.I))
            if main_content:
                paragraphs = main_content.find_all('p')
                content = ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 4: All paragraphs as fallback
        if not content or len(content) < 100:
            paragraphs = soup.find_all('p')
            content = ' '.join([p.get_text(strip=True) for p in paragraphs[:10]])
        
        # Clean content
        content = re.sub(r'\s+', ' ', content).strip()
        
        return content
    
    def scrape_arxiv(self, arxiv_id: str) -> Optional[Article]:
        """
        Specialized scraper for arXiv research papers
        
        Args:
            arxiv_id: arXiv ID (e.g., "2301.12345")
        """
        url = f"https://arxiv.org/abs/{arxiv_id}"
        
        try:
            response = requests.get(url, headers=self.headers, timeout=self.timeout)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Title
            title_tag = soup.find('h1', class_='title')
            title = title_tag.get_text(strip=True).replace('Title:', '').strip() if title_tag else ""
            
            # Abstract
            abstract_tag = soup.find('blockquote', class_='abstract')
            abstract = abstract_tag.get_text(strip=True).replace('Abstract:', '').strip() if abstract_tag else ""
            
            return Article(
                title=title,
                content=abstract,
                url=url,
                source='arxiv.org'
            )
            
        except Exception as e:
            logger.error(f"Failed to scrape arXiv {arxiv_id}: {e}")
            return None
    
    def scrape_multiple(self, urls: List[str], max_workers=5) -> List[Article]:
        """
        Scrape multiple URLs concurrently
        
        Args:
            urls: List of article URLs
            max_workers: Number of concurrent threads
        """
        articles = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_url = {executor.submit(self.scrape_url, url): url for url in urls}
            
            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    article = future.result()
                    if article:
                        articles.append(article)
                        logger.info(f"Successfully scraped: {article.title[:50]}...")
                except Exception as e:
                    logger.error(f"Error processing {url}: {e}")
        
        return articles



In [ ]:
class MultiDimensionalAnalyzer:
    """
    Analyzes articles across 5 dimensions
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embedding_model = SentenceTransformer(model_name)
        self.regression_model = MultiOutputRegressor(
            RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        )
        self.is_fitted = False
        
        self.dimensions = [
            'growth_potential',
            'recession_resistance',
            'automation_resistance',
            'skill_accessibility',
            'cross_industry_collaboration'
        ]
    
    def train(self, texts: List[str], scores_df: pd.DataFrame):
        """Train the model on labeled data"""
        logger.info("Creating embeddings for training data...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True, batch_size=32)
        
        scores_array = scores_df[self.dimensions].values
        
        logger.info("Training multi-output model...")
        self.regression_model.fit(embeddings, scores_array)
        self.is_fitted = True
        
        logger.info("Training complete!")
        return self
    
    def score_articles(self, articles: List[Article]) -> List[Article]:
        """
        Score multiple articles across all dimensions
        
        Args:
            articles: List of Article objects
            
        Returns:
            Same articles with scores populated
        """
        if not self.is_fitted:
            raise ValueError("Model must be trained first!")
        
        # Combine title and content for better context
        texts = [f"{art.title}. {art.content}" for art in articles]
        
        logger.info(f"Scoring {len(articles)} articles...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True, batch_size=32)
        predictions = self.regression_model.predict(embeddings)
        predictions = np.clip(predictions, 0, 10)
        
        # Add scores to articles
        for i, article in enumerate(articles):
            article.scores = {
                dim: float(predictions[i, j]) 
                for j, dim in enumerate(self.dimensions)
            }
        
        return articles


In [ ]:
class ArticlePipeline:
    """
    Complete pipeline: Scrape -> Analyze -> Store -> Recommend Majors
    """
    
    def __init__(self, analyzer: MultiDimensionalAnalyzer, scraper: Optional[ArticleScraper] = None):
        self.analyzer = analyzer
        self.scraper = scraper or ArticleScraper()
        self.vectorizedArticles: List[Dict] = []  # Main storage
        self.major_mapping = self._create_major_mapping()  # Map topics to majors
    
    def _create_major_mapping(self) -> Dict[str, List[str]]:
        """
        Map article topics/keywords to relevant college majors
        Customize this based on your needs
        """
        return {
            'computer_science': ['ai', 'artificial intelligence', 'machine learning', 'software', 
                                'programming', 'algorithm', 'data science', 'neural network', 
                                'deep learning', 'computer vision', 'nlp'],
            'electrical_engineering': ['chip', 'semiconductor', 'circuit', 'electronics', 
                                      'hardware', 'processor', 'quantum computing', 'robotics'],
            'mechanical_engineering': ['manufacturing', 'automotive', 'aerospace', 'materials', 
                                      'robotics', 'automation', '3d printing'],
            'biomedical_engineering': ['medical device', 'prosthetic', 'biomedical', 'healthcare technology',
                                      'medical imaging', 'biosensor'],
            'biology': ['genetics', 'crispr', 'biotech', 'gene therapy', 'synthetic biology',
                       'genomics', 'proteomics', 'microbiology'],
            'chemistry': ['drug discovery', 'pharmaceutical', 'materials science', 'catalyst',
                         'polymer', 'nanotechnology', 'battery'],
            'physics': ['quantum', 'particle', 'photonics', 'optics', 'laser', 'condensed matter'],
            'mathematics': ['cryptography', 'optimization', 'statistical', 'mathematical modeling',
                          'algorithms', 'theoretical computer science'],
            'economics': ['market', 'finance', 'trading', 'economic policy', 'inflation', 
                         'monetary', 'fiscal', 'econometrics'],
            'business': ['startup', 'entrepreneur', 'management', 'marketing', 'strategy',
                        'consulting', 'venture capital', 'business model'],
            'environmental_science': ['climate', 'renewable energy', 'solar', 'wind', 'sustainability',
                                     'carbon capture', 'green technology', 'environmental'],
            'neuroscience': ['brain', 'neural', 'cognitive', 'neuroscience', 'consciousness',
                           'neurotechnology', 'brain-computer interface'],
            'psychology': ['behavior', 'mental health', 'cognitive science', 'human factors',
                          'user experience', 'behavioral economics'],
            'data_science': ['data analytics', 'big data', 'data mining', 'visualization',
                           'predictive analytics', 'business intelligence'],
            'cybersecurity': ['security', 'encryption', 'hacking', 'cyber', 'vulnerability',
                            'network security', 'information security'],
            'political_science': ['policy', 'government', 'legislation', 'governance', 'politics',
                                'international relations', 'public policy']
        }
    
    def process_urls(self, urls: List[str], max_workers=5) -> pd.DataFrame:
        """
        Complete pipeline: scrape URLs, analyze, and store
        
        Args:
            urls: List of article URLs to scrape
            max_workers: Concurrent scraping threads
            
        Returns:
            DataFrame with results
        """
        logger.info(f"Starting pipeline for {len(urls)} URLs...")
        
        # Step 1: Scrape articles
        articles = self.scraper.scrape_multiple(urls, max_workers=max_workers)
        logger.info(f"Successfully scraped {len(articles)}/{len(urls)} articles")
        
        if not articles:
            logger.warning("No articles scraped successfully!")
            return pd.DataFrame()
        
        # Step 2: Score articles
        scored_articles = self.analyzer.score_articles(articles)
        
        # Step 3: Store in vectorizedArticles
        for article in scored_articles:
            self.vectorizedArticles.append({
                'title': article.title,
                'url': article.url,
                'source': article.source,
                **article.scores
            })
        
        # Return as DataFrame for easy viewing
        df = pd.DataFrame(self.vectorizedArticles)
        logger.info(f"Pipeline complete! Total articles in vectorizedArticles: {len(self.vectorizedArticles)}")
        
        return df
    
    def process_arxiv_papers(self, arxiv_ids: List[str]) -> pd.DataFrame:
        """Process arXiv papers specifically"""
        logger.info(f"Processing {len(arxiv_ids)} arXiv papers...")
        
        articles = []
        for arxiv_id in arxiv_ids:
            article = self.scraper.scrape_arxiv(arxiv_id)
            if article:
                articles.append(article)
            time.sleep(1)  # Respect rate limits
        
        logger.info(f"Successfully scraped {len(articles)}/{len(arxiv_ids)} papers")
        
        if articles:
            scored_articles = self.analyzer.score_articles(articles)
            
            for article in scored_articles:
                self.vectorizedArticles.append({
                    'title': article.title,
                    'url': article.url,
                    'source': article.source,
                    **article.scores
                })
        
        return pd.DataFrame(self.vectorizedArticles)
    
    def export_results(self, filename='article_scores.csv'):
        """Export vectorizedArticles to CSV"""
        df = pd.DataFrame(self.vectorizedArticles)
        df.to_csv(filename, index=False)
        logger.info(f"Exported {len(self.vectorizedArticles)} articles to {filename}")
        return df
    
    def get_top_articles(self, dimension: str, n=10) -> pd.DataFrame:
        """Get top N articles for a specific dimension"""
        df = pd.DataFrame(self.vectorizedArticles)
        return df.nlargest(n, dimension)[['title', 'source', dimension]]
    
    def _extract_article_topics(self, article: Dict) -> List[str]:
        """
        Extract relevant majors from article title and content
        Returns list of matching majors
        """
        text = (article.get('title', '') + ' ' + article.get('content', '')).lower()
        
        matching_majors = []
        for major, keywords in self.major_mapping.items():
            for keyword in keywords:
                if keyword.lower() in text:
                    matching_majors.append(major)
                    break  # One match per major is enough
        
        return matching_majors
    
    def recommend_major(self, 
                       student_interests: List[str],
                       student_priorities: Dict[str, float] = None,
                       top_n: int = 5,
                       use_semantic_matching: bool = True,
                       semantic_threshold: float = 0.3) -> pd.DataFrame:
        """
        Recommend majors for a student based on their interests and priorities
        
        Args:
            student_interests: List of topics/keywords OR characteristics
                              Topics: ['artificial intelligence', 'healthcare', 'robotics']
                              Characteristics: ['research focused', 'abstract thinking', 'human-element']
                              Mix: ['AI', 'helping people', 'creative problem solving']
            student_priorities: Dictionary mapping dimension to importance weight (0-1)
                               e.g., {'growth_potential': 1.0, 'recession_resistance': 0.7, ...}
                               If None, all dimensions weighted equally
            top_n: Number of major recommendations to return
            use_semantic_matching: If True, uses AI embeddings for matching (better for characteristics)
                                   If False, uses simple keyword matching (faster but less flexible)
            semantic_threshold: Similarity threshold for semantic matching (0-1, default 0.3)
        
        Returns:
            DataFrame with recommended majors, their scores, and supporting articles
        """
        if not self.vectorizedArticles:
            raise ValueError("No articles have been analyzed yet! Run process_urls() first.")
        
        # Default priorities if not specified
        if student_priorities is None:
            student_priorities = {
                'growth_potential': 1.0,
                'recession_resistance': 1.0,
                'automation_resistance': 1.0,
                'skill_accessibility': 1.0,
                'cross_industry_collaboration': 1.0
            }
        
        # Normalize weights
        total_weight = sum(student_priorities.values())
        normalized_priorities = {k: v / total_weight for k, v in student_priorities.items()}
        
        logger.info(f"Analyzing recommendations for student interested in: {student_interests}")
        logger.info(f"Priority weights: {normalized_priorities}")
        
        # Step 1: Filter articles matching student interests
        df = pd.DataFrame(self.vectorizedArticles)
        
        if use_semantic_matching:
            # SEMANTIC MATCHING: Works great for abstract characteristics
            logger.info("Using semantic matching (better for characteristics like 'research focused')")
            
            # Create embedding for student's combined interests/characteristics
            student_profile = ' '.join(student_interests)
            student_embedding = self.analyzer.embedding_model.encode([student_profile])[0]
            
            # Calculate similarity scores for each article
            def calculate_similarity(row):
                """Calculate semantic similarity between student profile and article"""
                article_text = str(row.get('title', '')) + ' ' + str(row.get('content', ''))
                article_embedding = self.analyzer.embedding_model.encode([article_text])[0]
                
                # Cosine similarity
                similarity = np.dot(student_embedding, article_embedding) / (
                    np.linalg.norm(student_embedding) * np.linalg.norm(article_embedding)
                )
                return similarity
            
            df['similarity_score'] = df.apply(calculate_similarity, axis=1)
            df['matches_interest'] = df['similarity_score'] >= semantic_threshold
            
            logger.info(f"Similarity scores range: {df['similarity_score'].min():.3f} to {df['similarity_score'].max():.3f}")
            
            relevant_articles = df[df['matches_interest']].copy()
            
            # Boost composite score by similarity
            if len(relevant_articles) > 0:
                logger.info(f"Found {len(relevant_articles)} articles with similarity >= {semantic_threshold}")
            else:
                # Lower threshold if no matches
                logger.warning(f"No matches at threshold {semantic_threshold}, lowering to 0.2")
                semantic_threshold = 0.2
                df['matches_interest'] = df['similarity_score'] >= semantic_threshold
                relevant_articles = df[df['matches_interest']].copy()
        else:
            # KEYWORD MATCHING: Faster but only works for explicit terms
            logger.info("Using keyword matching (better for specific topics like 'AI')")
            interest_keywords = [interest.lower() for interest in student_interests]
            
            def matches_interests(row):
                """Check if article matches any student interest"""
                text = (str(row.get('title', '')) + ' ' + str(row.get('content', ''))).lower()
                return any(keyword in text for keyword in interest_keywords)
            
            df['matches_interest'] = df.apply(matches_interests, axis=1)
            df['similarity_score'] = df['matches_interest'].astype(float)  # Binary 0 or 1
            relevant_articles = df[df['matches_interest']].copy()
        
        if len(relevant_articles) == 0:
            logger.warning("No articles match student interests. Using top 20% of articles by growth potential.")
            cutoff = df['growth_potential'].quantile(0.8)
            relevant_articles = df[df['growth_potential'] >= cutoff].copy()
            relevant_articles['similarity_score'] = 0.5  # Neutral similarity
        
        logger.info(f"Found {len(relevant_articles)} articles matching student interests")
        
        # Step 2: Calculate weighted composite score for each article
        dimensions = ['growth_potential', 'recession_resistance', 'automation_resistance', 
                     'skill_accessibility', 'cross_industry_collaboration']
        
        relevant_articles['composite_score'] = 0
        for dim in dimensions:
            if dim in normalized_priorities:
                relevant_articles['composite_score'] += (
                    relevant_articles[dim] * normalized_priorities[dim]
                )
        
        # Boost composite score by similarity (articles more aligned with student get higher scores)
        if use_semantic_matching:
            relevant_articles['composite_score'] = (
                relevant_articles['composite_score'] * (1 + relevant_articles['similarity_score'])
            )
        
        # Step 3: Map articles to majors
        relevant_articles['related_majors'] = relevant_articles.apply(
            lambda row: self._extract_article_topics(row.to_dict()), 
            axis=1
        )
        
        # Step 4: Aggregate scores by major
        major_scores = {}
        major_article_counts = {}
        major_top_articles = {}
        
        for _, article in relevant_articles.iterrows():
            majors = article['related_majors']
            score = article['composite_score']
            title = article['title']
            
            for major in majors:
                if major not in major_scores:
                    major_scores[major] = []
                    major_article_counts[major] = 0
                    major_top_articles[major] = []
                
                major_scores[major].append(score)
                major_article_counts[major] += 1
                major_top_articles[major].append({
                    'title': title,
                    'score': score,
                    'growth': article['growth_potential'],
                    'recession': article['recession_resistance'],
                    'automation': article['automation_resistance']
                })
        
        # Step 5: Calculate average scores and create recommendations
        recommendations = []
        for major, scores in major_scores.items():
            avg_score = np.mean(scores)
            max_score = np.max(scores)
            
            # Sort articles by score and get top 3
            top_articles = sorted(major_top_articles[major], 
                                key=lambda x: x['score'], 
                                reverse=True)[:3]
            
            recommendations.append({
                'major': major.replace('_', ' ').title(),
                'average_score': avg_score,
                'max_score': max_score,
                'num_articles': major_article_counts[major],
                'top_article': top_articles[0]['title'] if top_articles else '',
                'avg_growth': np.mean([a['growth'] for a in top_articles]) if top_articles else 0,
                'avg_recession_resistance': np.mean([a['recession'] for a in top_articles]) if top_articles else 0,
                'avg_automation_resistance': np.mean([a['automation'] for a in top_articles]) if top_articles else 0,
            })
        
        # Convert to DataFrame and sort
        recommendations_df = pd.DataFrame(recommendations)
        
        if len(recommendations_df) == 0:
            logger.warning("No major recommendations could be generated.")
            return pd.DataFrame()
        
        recommendations_df = recommendations_df.sort_values('average_score', ascending=False)
        
        logger.info(f"\nTop {top_n} Major Recommendations:")
        for idx, row in recommendations_df.head(top_n).iterrows():
            logger.info(f"  {row['major']}: {row['average_score']:.2f} "
                       f"(based on {row['num_articles']} articles)")
        
        return recommendations_df.head(top_n)
     def explain_recommendation(self, major: str) -> Dict:
        """
        Get detailed explanation for why a major was recommended
        
        Args:
            major: The major to explain (e.g., 'Computer Science')
        
        Returns:
            Dictionary with detailed breakdown
        """
        df = pd.DataFrame(self.vectorizedArticles)
        major_key = major.lower().replace(' ', '_')
        
        # Find articles related to this major
        def is_related(row):
            topics = self._extract_article_topics(row.to_dict())
            return major_key in topics
        
        df['is_related'] = df.apply(is_related, axis=1)
        related_articles = df[df['is_related']]
        
        if len(related_articles) == 0:
            return {'error': f'No articles found related to {major}'}
        
        # Calculate statistics
        dimensions = ['growth_potential', 'recession_resistance', 'automation_resistance',
                     'skill_accessibility', 'cross_industry_collaboration']
        
        stats = {
            'major': major,
            'num_articles': len(related_articles),
            'dimension_scores': {
                dim: {
                    'mean': float(related_articles[dim].mean()),
                    'median': float(related_articles[dim].median()),
                    'min': float(related_articles[dim].min()),
                    'max': float(related_articles[dim].max())
                }
                for dim in dimensions
            },
            'top_articles': related_articles.nlargest(5, 'growth_potential')[
                ['title', 'growth_potential', 'recession_resistance', 'automation_resistance']
            ].to_dict('records')
        }
        
        return stats


In [ ]:
if __name__ == "__main__":
    # Step 1: Train the analyzer on your labeled data
    print("=" * 80)
    print("STEP 1: Training the analyzer")
    print("=" * 80)
    
    # Load your labeled training data
    labeled_df = pd.read_csv('labeled_fields.csv')  # Your labeled dataset
    
    analyzer = MultiDimensionalAnalyzer()
    analyzer.train(
        texts=labeled_df['text'].tolist(),
        scores_df=labeled_df
    )
    
    # Step 2: Initialize the pipeline
    print("\n" + "=" * 80)
    print("STEP 2: Initializing pipeline")
    print("=" * 80)
    
    pipeline = ArticlePipeline(analyzer=analyzer)
    
    # Step 3: Scrape and analyze articles
    print("\n" + "=" * 80)
    print("STEP 3: Scraping and analyzing articles")
    print("=" * 80)
    
    # Example URLs (replace with your target sources)
    news_urls = [
        'https://www.nature.com/articles/d41586-024-00001-x',
        'https://techcrunch.com/2024/01/15/ai-breakthrough/',
        'https://www.theverge.com/23950084/quantum-computing-breakthrough',
        # Add more URLs here
    ]
    
    # Process news articles
    results_df = pipeline.process_urls(news_urls, max_workers=5)
    print("\n" + results_df.to_string())
    
    # Process arXiv papers
    arxiv_ids = ['2401.12345', '2401.67890']  # Example arXiv IDs
    arxiv_df = pipeline.process_arxiv_papers(arxiv_ids)
    
    # Step 4: Access vectorizedArticles
    print("\n" + "=" * 80)
    print("STEP 4: Accessing vectorizedArticles")
    print("=" * 80)
    
    print(f"\nTotal articles processed: {len(pipeline.vectorizedArticles)}")
    print("\nFirst 3 articles in vectorizedArticles:")
    for i, article in enumerate(pipeline.vectorizedArticles[:3]):
        print(f"\n{i+1}. {article['title']}")
        print(f"   Growth: {article['growth_potential']:.1f}, "
              f"Recession: {article['recession_resistance']:.1f}, "
              f"Automation: {article['automation_resistance']:.1f}")
    
    # Step 5: MAJOR RECOMMENDATION SYSTEM
    print("\n" + "=" * 80)
    print("STEP 5: MAJOR RECOMMENDATIONS FOR STUDENTS")
    print("=" * 80)
    
    # Example Student 1: Interested in AI and healthcare
    print("\n--- Student 1: AI & Healthcare Enthusiast (Topic-based) ---")
    student1_interests = ['artificial intelligence', 'machine learning', 'healthcare', 'medical']
    student1_priorities = {
        'growth_potential': 1.0,          # Cares most about growth
        'recession_resistance': 0.8,      # Also values stability
        'automation_resistance': 0.5,     # Less concerned about automation
        'skill_accessibility': 0.3,       # Willing to do hard degrees
        'cross_industry_collaboration': 0.7
    }
    
    recommendations1 = pipeline.recommend_major(
        student_interests=student1_interests,
        student_priorities=student1_priorities,
        top_n=5,
        use_semantic_matching=True  # Use semantic matching
    )
    print("\n" + recommendations1.to_string())
    
    # Example Student 2: Characteristics-based (abstract traits)
    print("\n\n--- Student 2: Research-Focused Abstract Thinker ---")
    student2_interests = [
        'research focused', 
        'abstract thinking', 
        'theoretical work',
        'solving complex problems',
        'independent study'
    ]
    student2_priorities = {
        'growth_potential': 1.0,
        'recession_resistance': 0.6,
        'automation_resistance': 0.8,      # Values non-automatable work
        'skill_accessibility': 0.4,        # Willing to pursue advanced degrees
        'cross_industry_collaboration': 0.5
    }
    
    recommendations2 = pipeline.recommend_major(
        student_interests=student2_interests,
        student_priorities=student2_priorities,
        top_n=5,
        use_semantic_matching=True,        # IMPORTANT: Use semantic for characteristics
        semantic_threshold=0.25             # Lower threshold for abstract characteristics
    )
    print("\n" + recommendations2.to_string())
    
    # Example Student 3: Human-centered characteristics
    print("\n\n--- Student 3: People-Oriented Practical Problem Solver ---")
    student3_interests = [
        'human element',
        'helping people',
        'hands-on work',
        'real-world impact',
        'working with communities',
        'practical applications'
    ]
    student3_priorities = {
        'growth_potential': 0.7,
        'recession_resistance': 1.0,       # Wants job security
        'automation_resistance': 1.0,      # Wants human-centered work
        'skill_accessibility': 0.8,        # Prefers accessible entry
        'cross_industry_collaboration': 0.9
    }
    
    recommendations3 = pipeline.recommend_major(
        student_interests=student3_interests,
        student_priorities=student3_priorities,
        top_n=5,
        use_semantic_matching=True,
        semantic_threshold=0.25
    )
    print("\n" + recommendations3.to_string())
    
    # Example Student 4: Mix of topics and characteristics
    print("\n\n--- Student 4: Creative Tech Innovator (Mixed Input) ---")
    student4_interests = [
        'technology',                       # Topic
        'creative problem solving',         # Characteristic
        'innovation',                       # Characteristic
        'design',                          # Topic
        'user experience',                 # Topic
        'interdisciplinary thinking'        # Characteristic
    ]
    student4_priorities = {
        'growth_potential': 1.0,
        'recession_resistance': 0.5,
        'automation_resistance': 0.7,
        'skill_accessibility': 0.9,
        'cross_industry_collaboration': 1.0
    }
    
    recommendations4 = pipeline.recommend_major(
        student_interests=student4_interests,
        student_priorities=student4_priorities,
        top_n=5,
        use_semantic_matching=True
    )
    print("\n" + recommendations4.to_string())
    
    # Step 6: Detailed explanation for a specific major
    print("\n" + "=" * 80)
    print("STEP 6: DETAILED MAJOR EXPLANATION")
    print("=" * 80)
    
    if len(recommendations1) > 0:
        top_major = recommendations1.iloc[0]['major']
        explanation = pipeline.explain_recommendation(top_major)
        
        print(f"\nDetailed Analysis for {top_major}:")
        print(f"Based on {explanation['num_articles']} articles")
        print("\nDimension Scores:")
        for dim, scores in explanation['dimension_scores'].items():
            print(f"  {dim}: Mean={scores['mean']:.2f}, Range=[{scores['min']:.1f}, {scores['max']:.1f}]")
        
        print("\nTop Supporting Articles:")
        for i, article in enumerate(explanation['top_articles'][:3], 1):
            print(f"  {i}. {article['title'][:60]}...")
            print(f"     Growth: {article['growth_potential']:.1f}, "
                  f"Recession: {article['recession_resistance']:.1f}")
    
    # Step 7: Export results
    print("\n" + "=" * 80)
    print("STEP 7: EXPORT RESULTS")
    print("=" * 80)
    
    # Export article scores
    pipeline.export_results('article_analysis_results.csv')
    
    # Export recommendations
    all_recommendations = pd.concat([recommendations1, recommendations2, recommendations3, recommendations4], 
                                   ignore_index=True)
    all_recommendations.to_csv('student_major_recommendations.csv', index=False)
    print("Saved recommendations to 'student_major_recommendations.csv'")
    
    # Access the vectorizedArticles variable directly
    vectorizedArticles = pipeline.vectorizedArticles
    print(f"\nvectorizedArticles contains {len(vectorizedArticles)} articles")
    
    print("\n" + "=" * 80)
    print("QUICK USAGE SUMMARY")
    print("=" * 80)
    print("""
# 1. Train analyzer
analyzer = MultiDimensionalAnalyzer()
analyzer.train(texts, scores_df)

# 2. Create pipeline and scrape articles
pipeline = ArticlePipeline(analyzer)
pipeline.process_urls(['url1', 'url2', ...])

# 3. Get recommendations for a student with TOPICS
recommendations = pipeline.recommend_major(
    student_interests=['AI', 'healthcare', 'robotics'],
    student_priorities={'growth_potential': 1.0, ...},
    use_semantic_matching=True  # Better for characteristics
)

# 4. Get recommendations for a student with CHARACTERISTICS
recommendations = pipeline.recommend_major(
    student_interests=['research focused', 'abstract thinking', 'human element'],
    student_priorities={'automation_resistance': 1.0, ...},
    use_semantic_matching=True,     # REQUIRED for characteristics
    semantic_threshold=0.25          # Lower threshold for abstract traits
)

# 5. Mix topics and characteristics
recommendations = pipeline.recommend_major(
    student_interests=['technology', 'creative problem solving', 'helping people'],
    use_semantic_matching=True
)

# 6. View results
print(recommendations)
    """)